> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# We ran the AI trading desk on nothing: every number

Education about methods, not financial advice. A viral article builds a trading desk out of AI agents (scout, chart analyst, strategist, risk officer, trader, chief of staff) and gives the strategist three rejection rules. Here we build that checklist as written and feed it **500 made-up stocks with no edge at all**: zero drift, fat tails, volume that rises on big days and predicts nothing. Whatever the desk approves, it approved from noise.

The code is in `desk_sim.py`; this notebook recomputes each number in the video.

HYPOTHETICAL / SIMULATED RESULTS. MIT License.

The next cell writes the helper module `desk_sim.py` next to this notebook: the same code the animations use. Run it first.

In [ ]:
%%writefile desk_sim.py
# MIT License - Copyright (c) 2026 QuantWall
# Full text: https://opensource.org/license/mit
# HYPOTHETICAL / SIMULATED RESULTS - EDUCATIONAL ONLY - NOT FINANCIAL ADVICE
"""Every number in "We Fed a Viral 'AI Trading Desk' Fake Markets" (desk_on_noise.ipynb recomputes them).

A viral article builds a team of AI agents into a trading desk: a Scout finds what is moving, a Chart
Analyst scores the setup, a Strategist writes a plan and backtests it on three years with fees, and
rejects it if it has fewer than 30 trades, fails on the most recent year, or only works with one exact
setting. Then two weeks of paper trading must "match the backtest", and a weekly review pauses
strategies drifting below expectations.

Here every market is made up and has NO edge: zero drift, fat tails, volume that tracks the size of the
move but predicts nothing. Whatever the desk approves, it approved from noise.

Run:  python desk_sim.py 32  (32 desk-years, writes desk_results.json next to this file)
"""

import json
import os
import sys
from itertools import product
from multiprocessing import Pool

import numpy as np

DAYS = 252
N_STOCKS = 500
YEARS = 6                 # 3 years of history, 1 desk year, 1 forward year, spare
HIST = 3 * DAYS           # the Strategist's backtest window
DESK0 = 3 * DAYS + 210    # first desk day (history plus room for the 200-day average)
FEE = 0.0005              # fees + slippage per side (5 bp), 10 bp a round trip
SCOUT_TOP = 5             # Scout sends the top 5 movers a day
SCORE_MIN = 7             # Chart Analyst passes setups scoring 7 or more


# ----------------------------------------------------------------------------- the made-up market
def universe(seed):
    rng = np.random.default_rng(seed)
    T = YEARS * DAYS
    sig = rng.uniform(0.012, 0.030, size=N_STOCKS)                 # 19% to 48% a year
    z = rng.standard_t(4, size=(T, N_STOCKS)) / np.sqrt(2.0)        # unit variance, fat tails
    r = z * sig                                                     # zero mean: no edge anywhere
    vol = np.exp(0.35 * rng.standard_normal((T, N_STOCKS)) + 0.6 * np.abs(z))   # big move -> big volume
    return r, vol, sig


def rolling_max_prev(x, n):
    """max of the previous n values (excluding today); nan for the first n."""
    out = np.full_like(x, np.nan)
    from numpy.lib.stride_tricks import sliding_window_view
    w = sliding_window_view(x, n, axis=0).max(axis=-1)              # w[i] = max x[i..i+n-1]
    out[n:] = w[:-1]
    return out


def rolling_mean(x, n):
    c = np.concatenate([np.zeros((1,) + x.shape[1:]), np.cumsum(x, axis=0)])
    out = np.full_like(x, np.nan)
    out[n - 1:] = (c[n:] - c[:-n]) / n
    return out


def rsi(logp, n):
    d = np.diff(logp, axis=0, prepend=logp[:1])
    up, dn = np.clip(d, 0, None), np.clip(-d, 0, None)
    a = 1.0 / n
    au = np.zeros_like(d)
    ad = np.zeros_like(d)
    au[0], ad[0] = up[0], dn[0]
    for t in range(1, len(d)):
        au[t] = (1 - a) * au[t - 1] + a * up[t]
        ad[t] = (1 - a) * ad[t - 1] + a * dn[t]
    return 100 - 100 / (1 + au / np.maximum(ad, 1e-12))


class Market:
    def __init__(self, seed):
        self.r, self.vol, self.sig = universe(seed)
        self.p = np.cumsum(self.r, axis=0)                          # log price
        self.T = self.r.shape[0]
        self.ma = {L: rolling_mean(self.p, L) for L in (20, 50, 100, 200)}
        self.vavg = rolling_mean(self.vol, 20)
        self.atr = rolling_mean(np.abs(self.r), 14)
        self.rsi = {n: rsi(self.p, n) for n in (2, 5, 14)}
        self.hi = {}

    def high(self, n):
        if n not in self.hi:
            self.hi[n] = rolling_max_prev(self.p, n)
        return self.hi[n]

    def low(self, n):
        k = -n
        if k not in self.hi:
            self.hi[k] = -rolling_max_prev(-self.p, n)
        return self.hi[k]


# ----------------------------------------------------------------------------- the desk's agents
def scout(m, t):
    """The five biggest moves today, in units of each stock's own swing (the 'story')."""
    s = np.abs(m.r[t]) / m.sig
    return np.argsort(-s)[:SCOUT_TOP]


def chart_score(m, i, t, d):
    """1 to 10, as the article's checklist: trend, averages, RSI, volume, breakout. d = +1 long, -1 short."""
    p = m.p[t, i]
    s = 1
    s += 2 * (d * (p - m.ma[50][t, i]) > 0)
    s += 2 * (d * (m.ma[50][t, i] - m.ma[200][t, i]) > 0)
    s += 2 * (d * (m.rsi[14][t, i] - 50) > 0)
    s += 2 * (m.vol[t, i] > 1.5 * m.vavg[t, i])
    s += 1 * ((p > m.high(20)[t, i]) if d > 0 else (p < m.low(20)[t, i]))
    return int(s)


# A plan = (kind, knobs). Knob grids are the steps the 'one exact setting' check moves along.
GRIDS = {
    "breakout": {"N": [10, 15, 20, 30, 40, 55], "H": [2, 3, 4, 5, 7, 10], "stop": [1.0, 1.5, 2.0, 3.0]},
    "pullback": {"L": [50, 100, 200], "n": [2, 5, 14], "x": [25, 30, 35, 40, 45], "H": [2, 3, 4, 5, 7, 10]},
    "follow":   {"m": [1.5, 2.0, 2.5, 3.0], "H": [2, 3, 4, 5, 7, 10], "stop": [1.0, 1.5, 2.0, 3.0]},
}


def random_plan(rng):
    kind = list(GRIDS)[rng.integers(3)]
    knobs = {k: int(rng.integers(len(v))) for k, v in GRIDS[kind].items()}
    return kind, knobs


def neighbours(plan):
    """Every setting one step away on one knob (the 'only works with one exact setting' check)."""
    kind, knobs = plan
    out = []
    for k, v in GRIDS[kind].items():
        for step in (-1, 1):
            j = knobs[k] + step
            if 0 <= j < len(v):
                nk = dict(knobs)
                nk[k] = j
                out.append((kind, nk))
    return out


def entries(m, i, plan, d, t0, t1):
    """Boolean entry signals on days t0..t1-1 for stock i (decided at that day's close)."""
    kind, kn = plan
    g = {k: GRIDS[kind][k][j] for k, j in kn.items()}
    sl = slice(t0, t1)
    p = m.p[sl, i]
    if kind == "breakout":
        lvl = m.high(g["N"])[sl, i] if d > 0 else m.low(g["N"])[sl, i]
        sig = (d * (p - lvl) > 0) & (m.vol[sl, i] > 1.2 * m.vavg[sl, i])
    elif kind == "pullback":
        trend = d * (p - m.ma[g["L"]][sl, i]) > 0
        x = g["x"]
        stretch = (m.rsi[g["n"]][sl, i] < x) if d > 0 else (m.rsi[g["n"]][sl, i] > 100 - x)
        sig = trend & stretch
    else:
        sig = d * m.r[sl, i] > g["m"] * m.sig[i]
    return np.nan_to_num(sig, nan=0).astype(bool), g


def backtest(m, i, plan, d, t0, t1):
    """Trades on days t0..t1. Enter at the next close after a signal, exit after H days or a stop
    (k x 14-day average move, on closes). One position at a time. Net of 10 bp a round trip.
    Returns (list of (entry_day, net trade return))."""
    sig, g = entries(m, i, plan, d, t0, t1)
    H = g["H"]
    k = g.get("stop", None)
    r = m.r[:, i]
    trades = []
    t = 0
    n = len(sig)
    while t < n - 1:
        if sig[t]:
            e = t0 + t + 1                                   # first day held: the day after the signal
            stop = -k * m.atr[t0 + t, i] * np.sqrt(H) if k else -np.inf
            cum = 0.0
            h = 0
            while h < H and e + h < t0 + n:
                cum += d * r[e + h]
                h += 1
                if cum <= stop:
                    break
            trades.append((e, cum - 2 * FEE))
            t = t + h + 1
        else:
            t += 1
    return trades


def strategist(m, i, plan, d, t):
    """The article's three rejection rules, on the three years up to the idea's day."""
    tr = backtest(m, i, plan, d, t - HIST, t + 1)
    x = np.array([v for _, v in tr])
    n = len(x)
    rep = {"n": n, "total": float(x.sum()) if n else 0.0}
    last = np.array([v for e, v in tr if e > t - DAYS])
    rep["last"] = float(last.sum()) if len(last) else 0.0
    rep["tstat"] = float(x.mean() / x.std(ddof=1) * np.sqrt(n)) if n > 2 and x.std() > 0 else 0.0
    rep["mean"] = float(x.mean()) if n else 0.0
    rep["sd"] = float(x.std(ddof=1)) if n > 2 else 0.0
    rep["win"] = float((x > 0).mean()) if n else 0.0
    rep["pass_n"] = n >= 30
    rep["pass_total"] = rep["pass_n"] and rep["total"] > 0
    rep["pass_last"] = rep["pass_total"] and rep["last"] > 0
    if rep["pass_last"]:
        nb = [sum(v for _, v in backtest(m, i, q, d, t - HIST, t + 1)) for q in neighbours(plan)]
        rep["nb_pos"] = float(np.mean(np.array(nb) > 0))
        rep["pass_nb"] = bool(np.all(np.array(nb) > 0))
        rep["pass_nb_major"] = rep["nb_pos"] >= 0.5
    else:
        rep["pass_nb"] = rep["pass_nb_major"] = False
    return rep


def forward(m, i, plan, d, t, days):
    """The approved plan traded on the days after approval."""
    t1 = min(t + 1 + days, m.T)
    return backtest(m, i, plan, d, t + 1, t1)


# ----------------------------------------------------------------------------- one desk year
def desk_year(seed):
    m = Market(seed)
    rng = np.random.default_rng(10_000 + seed)
    rows = []
    for t in range(DESK0, DESK0 + DAYS):
        for i in scout(m, t):
            d = 1 if m.r[t, i] > 0 else -1
            sc = chart_score(m, i, t, d)
            row = {"t": t, "i": int(i), "d": d, "score": sc}
            if sc >= SCORE_MIN:
                plan = random_plan(rng)
                rep = strategist(m, i, plan, d, t)
                row.update(rep)
                row["plan"] = plan
                if rep["pass_nb"] or rep["pass_nb_major"]:
                    fw = forward(m, i, plan, d, t, DAYS)
                    row["fwd"] = [v for _, v in fw]
                    row["fwd_days"] = [e - t for e, _ in fw]
            rows.append(row)
    return rows


def summarise(all_rows, n_years):
    ideas = [r for r in all_rows if "n" in r]
    S = {}
    S["desk_years"] = n_years
    S["scout_per_year"] = len(all_rows) / n_years
    S["to_strategist_per_year"] = len(ideas) / n_years
    S["to_strategist_per_day"] = len(ideas) / n_years / DAYS
    for key in ("pass_n", "pass_total", "pass_last", "pass_nb", "pass_nb_major"):
        S[key + "_per_year"] = sum(r[key] for r in ideas) / n_years
        S[key + "_share"] = sum(r[key] for r in ideas) / len(ideas)
    # conditional pass rates (each rule given the previous ones)
    S["cond_total_given_n"] = S["pass_total_share"] / S["pass_n_share"]
    S["cond_last_given_total"] = S["pass_last_share"] / S["pass_total_share"]
    S["cond_nb_given_last"] = S["pass_nb_share"] / S["pass_last_share"]
    S["cond_nbmaj_given_last"] = S["pass_nb_major_share"] / S["pass_last_share"]
    # independent-looking? positive last year among all >=30-trade plans
    n30 = [r for r in ideas if r["pass_n"]]
    S["last_pos_given_n_share"] = float(np.mean([r["last"] > 0 for r in n30]))
    S["total_pos_given_n_share"] = float(np.mean([r["total"] > 0 for r in n30]))
    S["last_pos_given_total_neg"] = float(np.mean([r["last"] > 0 for r in n30 if r["total"] <= 0]))
    # the approved plans
    for tag, key in (("strict", "pass_nb"), ("majority", "pass_nb_major")):
        ap = [r for r in ideas if r[key]]
        bt = np.array([r["tstat"] for r in ap])
        S[f"{tag}_n_approved"] = len(ap)
        S[f"{tag}_median_trades"] = float(np.median([r["n"] for r in ap]))
        S[f"{tag}_median_win"] = float(np.median([r["win"] for r in ap]))
        S[f"{tag}_median_tstat"] = float(np.median(bt))
        S[f"{tag}_share_t2"] = float(np.mean(bt >= 2))
        S[f"{tag}_bt_mean_trade_pct"] = float(100 * np.mean([r["mean"] for r in ap]))
        f = np.concatenate([r["fwd"] for r in ap if r["fwd"]])
        S[f"{tag}_fwd_trades"] = int(len(f))
        S[f"{tag}_fwd_mean_trade_pct"] = float(100 * f.mean())
        S[f"{tag}_fwd_win"] = float((f > 0).mean())
        S[f"{tag}_fwd_t"] = float(f.mean() / f.std(ddof=1) * np.sqrt(len(f)))
        S[f"{tag}_fwd_plan_pos_share"] = float(np.mean([sum(r["fwd"]) > 0 for r in ap if r["fwd"]]))
        # paper rung: first 10 trading days
        n2w = np.array([sum(1 for dd in r["fwd_days"] if dd <= 10) for r in ap])
        S[f"{tag}_paper2w_trades_mean"] = float(n2w.mean())
        S[f"{tag}_paper2w_zero_share"] = float(np.mean(n2w == 0))
        S[f"{tag}_paper2w_le1_share"] = float(np.mean(n2w <= 1))
        inside = []
        for r, k in zip(ap, n2w):
            pnl = sum(v for v, dd in zip(r["fwd"], r["fwd_days"]) if dd <= 10)
            lo = k * r["mean"] - 1.96 * r["sd"] * np.sqrt(k)
            hi = k * r["mean"] + 1.96 * r["sd"] * np.sqrt(k)
            inside.append(lo <= pnl <= hi)
        S[f"{tag}_paper2w_inside_share"] = float(np.mean(inside))
        # weekly review: pause any plan whose live P&L falls below its expected path minus one swing
        alive_tot, all_tot = [], []
        for r in ap:
            pnl, paused = 0.0, False
            kept = []
            for v, dd in zip(r["fwd"], r["fwd_days"]):
                if paused:
                    break
                kept.append(v)
                pnl += v
                k = len(kept)
                if k >= 2 and pnl < k * r["mean"] - r["sd"] * np.sqrt(k):
                    paused = True
            all_tot.append(sum(r["fwd"]))
            if not paused:
                alive_tot.append(pnl)
        S[f"{tag}_review_alive_share"] = len(alive_tot) / len(ap)
        S[f"{tag}_review_alive_mean_pct"] = float(100 * np.mean(alive_tot))
        S[f"{tag}_review_all_mean_pct"] = float(100 * np.mean(all_tot))
        S[f"{tag}_review_alive_pos_share"] = float(np.mean(np.array(alive_tot) > 0))
    # what an honest single bar would have let through (t >= 2 on the backtest, nothing else)
    S["t2_share_of_ideas"] = float(np.mean([r["tstat"] >= 2 for r in ideas]))
    S["t2_and_30_share_of_ideas"] = float(np.mean([r["tstat"] >= 2 and r["n"] >= 30 for r in ideas]))
    # an example: approved, ~47 trades
    ex = sorted([r for r in ideas if r["pass_nb"]], key=lambda r: abs(r["n"] - 47))[:1]
    if ex:
        e = ex[0]
        S["example"] = {k: e[k] for k in ("n", "total", "last", "tstat", "win", "mean", "sd")}
        S["example"]["fwd_total"] = float(sum(e["fwd"]))
        S["example"]["fwd_n"] = len(e["fwd"])
    return S


def viz(n_years=32):
    """Small data for the scenes: the desk year whose strict approvals are closest to the average,
    its approved plans' next year (trade day, return), the weekly-review pause week, and the example
    plan's trades next to one neighbour's on the same stock."""
    with Pool(min(16, n_years)) as pool:
        out = pool.map(desk_year, range(1, n_years + 1))
    counts = [sum(r.get("pass_nb", False) for r in yr) for yr in out]
    avg = np.mean(counts)
    k = int(np.argmin([abs(c - avg) for c in counts]))
    seed = k + 1
    ap = [r for r in out[k] if r.get("pass_nb")]
    plans = []
    for r in ap:
        pnl, pause = 0.0, None
        for j, (v, dd) in enumerate(zip(r["fwd"], r["fwd_days"])):
            pnl += v
            n = j + 1
            if n >= 2 and pnl < n * r["mean"] - r["sd"] * np.sqrt(n):
                pause = int(np.ceil(dd / 5))      # the Sunday after
                break
        plans.append({"days": r["fwd_days"], "ret": r["fwd"], "pause_week": pause, "bt_mean": r["mean"],
                      "bt_n": r["n"], "approved_day": r["t"] - DESK0})
    # example: approved plan nearest 47 trades across all years, its trades and a neighbour's
    allap = [(yi, r) for yi, yr in enumerate(out) for r in yr if r.get("pass_nb")]
    yi, ex = min(allap, key=lambda z: abs(z[1]["n"] - 47))
    m = Market(yi + 1)
    t = ex["t"]
    mine = backtest(m, ex["i"], ex["plan"], ex["d"], t - HIST, t + 1)
    nbs = neighbours(ex["plan"])
    best = None
    for q in nbs:
        tq = backtest(m, ex["i"], q, ex["d"], t - HIST, t + 1)
        a = {e for e, _ in mine}
        b = {e for e, _ in tq}
        share = len(a & b) / max(1, len(a | b))
        if best is None or share > best[0]:
            best = (share, q, tq)
    price = np.exp(m.p[t - HIST:t + 1, ex["i"]] - m.p[t - HIST, ex["i"]])
    data = {"seed": seed, "counts": counts, "plans": plans,
            "example": {"plan": ex["plan"], "n": ex["n"], "total": ex["total"], "last": ex["last"], "tstat": ex["tstat"],
                        "win": ex["win"], "fwd": ex["fwd"], "fwd_days": ex["fwd_days"],
                        "price": [float(x) for x in price[::3]],
                        "mine": [[e - (t - HIST), v] for e, v in mine],
                        "neighbour": {"plan": best[1], "share": best[0], "trades": [[e - (t - HIST), v] for e, v in best[2]]}}}
    here = os.path.dirname(os.path.abspath(__file__))
    with open(os.path.join(here, "desk_viz.json"), "w") as f:
        json.dump(data, f)
    print("seed", seed, "approved", len(ap), "avg", avg, "share", best[0], "example", ex["n"], ex["plan"], best[1])


def main():
    n_years = int(sys.argv[1]) if len(sys.argv) > 1 else 16
    with Pool(min(16, n_years)) as pool:
        out = pool.map(desk_year, range(1, n_years + 1))
    rows = [r for yr in out for r in yr]
    S = summarise(rows, n_years)
    here = os.path.dirname(os.path.abspath(__file__))
    with open(os.path.join(here, "desk_results.json"), "w") as f:
        json.dump(S, f, indent=1)
    print(json.dumps(S, indent=1))


if __name__ == "__main__":
    for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[v] = "1"
    if len(sys.argv) > 1 and sys.argv[1] == "viz":
        viz()
    else:
        main()

## 1. Run the desk

Each desk-year draws a fresh made-up market (seed 1, 2, 3, ...) and runs the desk for one year: every day the scout picks the five biggest movers, the chart analyst scores each one from 1 to 10, and every setup scoring 7 or more goes to the strategist, who writes a plan and backtests it on the stock's last three years at 0.10% a round trip.

The video averages **32 desk-years**. On a laptop with 8 cores that takes several minutes; set `YEARS = 8` for a quicker look (the numbers move a little, the story doesn't).

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
from multiprocessing import Pool
import numpy as np
import desk_sim as D

YEARS = 32
with Pool(min(os.cpu_count() or 1, YEARS)) as pool:
    out = pool.map(D.desk_year, range(1, YEARS + 1))
rows = [r for yr in out for r in yr]
S = D.summarise(rows, YEARS)
print(f"{YEARS} desk-years done")

## 2. The funnel: how many ideas survive each rule, per year

The strict reading of "only works with one exact setting": every setting one step away on any knob must also be profitable. The looser reading (half of them) is printed too.

In [ ]:
stages = [("scout's stories", S["scout_per_year"]),
          ("chart analyst passes (7+)", S["to_strategist_per_year"]),
          ("30+ trades", S["pass_n_per_year"]),
          ("profitable over 3 years", S["pass_total_per_year"]),
          ("profitable in the last year", S["pass_last_per_year"]),
          ("every nearby setting profitable", S["pass_nb_per_year"])]
for name, v in stages:
    print(f"{name:34} {v:7.0f}")
print(f"\nideas per day reaching the strategist: {S['to_strategist_per_day']:.1f}")
print(f"rejected: {100 * (1 - S['pass_nb_share']):.0f}%   approved: {100 * S['pass_nb_share']:.1f}%")
print(f"looser neighbour check (half the neighbours): {S['pass_nb_major_per_year']:.0f} approved a year")

## 3. Why the rules don't bite

**The last year is not a second test.** It sits inside the three-year backtest, so a plan that made money over three years usually made money in its last year too.

In [ ]:
n30 = [r for r in rows if r.get("pass_n")]
pos = np.mean([r["last"] > 0 for r in n30 if r["total"] > 0])
neg = np.mean([r["last"] > 0 for r in n30 if r["total"] <= 0])
print(f"last year profitable when the 3 years were:     {100 * pos:.0f}%")
print(f"last year profitable when the 3 years were not: {100 * neg:.0f}%")

**Every rule checks the sign, not the size.** Profitable, profitable last year, neighbours profitable: none asks whether the profit is bigger than luck would give.

In [ ]:
print(f"approved plans: median {S['strict_median_trades']:.0f} trades, {100 * S['strict_median_win']:.0f}% winners, "
      f"median t-statistic {S['strict_median_tstat']:.1f}")
print(f"share of approved plans with a t-statistic of 2 or more: {100 * S['strict_share_t2']:.0f}%")
e = S["example"]
print(f"\nexample approved plan: {e['n']} trades, {100 * e['total']:+.0f}% over 3 years, {100 * e['last']:+.0f}% last year, "
      f"{100 * e['win']:.0f}% winners, t-statistic {e['tstat']:.1f}")
print(f"its next year: {e['fwd_n']} trades, {100 * e['fwd_total']:+.1f}%")

**Neighbouring settings take the same trades.** For the example plan, the closest neighbour (one knob one step away) enters on almost the same days, so "it still works one step away" is the same result counted twice.

In [ ]:
allap = [(yi, r) for yi, yr in enumerate(out) for r in yr if r.get("pass_nb")]
yi, ex = min(allap, key=lambda z: abs(z[1]["n"] - 47))
m = D.Market(yi + 1)
t = ex["t"]
mine = {e for e, _ in D.backtest(m, ex["i"], ex["plan"], ex["d"], t - D.HIST, t + 1)}
def entry_days(q):
    return {e for e, _ in D.backtest(m, ex["i"], q, ex["d"], t - D.HIST, t + 1)}
best = max(D.neighbours(ex["plan"]), key=lambda q: len(mine & entry_days(q)) / len(mine | entry_days(q)))
theirs = entry_days(best)
print("plan:", ex["plan"], "  neighbour:", best)
print(f"same entry days: {100 * len(mine & theirs) / len(mine | theirs):.0f}%")

## 4. Backtest vs the next year

Each approved plan is traded exactly as written for the year after it was approved.

In [ ]:
print(f"backtest:  {S['strict_bt_mean_trade_pct']:+.2f}% a trade")
print(f"next year: {S['strict_fwd_mean_trade_pct']:+.2f}% a trade over {S['strict_fwd_trades']} trades, "
      f"{100 * S['strict_fwd_win']:.0f}% winners, {100 * S['strict_fwd_plan_pos_share']:.0f}% of plans profitable")
print("(a round trip costs 0.10%: the next year is about the cost of trading)")

## 5. Two weeks of paper trading

"Paper matches backtest": the plan's first ten trading days, judged against the backtest's own range (mean per trade times the trades taken, plus or minus 1.96 swings).

In [ ]:
print(f"trades in the first two weeks, on average: {S['strict_paper2w_trades_mean']:.1f}")
print(f"no trade at all:   {100 * S['strict_paper2w_zero_share']:.0f}%")
print(f"one trade or none: {100 * S['strict_paper2w_le1_share']:.0f}%")
print(f"'inside the expected range': {100 * S['strict_paper2w_inside_share']:.0f}%")

How long would it take to tell the median approved plan from zero, even if its backtest were true? A t-statistic grows with the square root of the number of trades.

In [ ]:
per_trade = S["strict_median_tstat"] / np.sqrt(S["strict_median_trades"])
need = (2 / per_trade) ** 2
per_year = S["strict_median_trades"] / 3
print(f"per-trade signal-to-noise: {per_trade:.3f}")
print(f"trades needed for a t-statistic of 2: {need:.0f}  (about {need / per_year:.0f} years at {per_year:.0f} trades a year)")

## 6. The weekly review

Pause a plan when its live total falls more than one swing below its expected path. The ones still running look great, because the review removed the others.

In [ ]:
print(f"still running after a year: {100 * S['strict_review_alive_share']:.0f}% of approved plans")
print(f"their average result: {S['strict_review_alive_mean_pct']:+.1f}%   every approved plan: {S['strict_review_all_mean_pct']:+.1f}%")
print(f"running plans that are profitable: {100 * S['strict_review_alive_pos_share']:.0f}%")

## 7. The noise officer

Everything above is one check the desk is missing: run the whole desk on markets with nothing in them and count its yeses. If it says yes about forty times a year to nothing, one more yes on real data tells you very little, however clean the journal looks. Swap in your own rules in `desk_sim.py` and run this notebook again.